# Ranking Comparativo de Modelos de Machine Learning

Este notebook gera dinamicamente um ranking de todos os modelos treinados, agrupados por:
- **Tipo de problema**: Regressão vs Classificação
- **Janela de tempo**: 3, 7, 15 e 30 dias

Os dados são carregados automaticamente a partir dos CSVs de métricas gerados no treinamento.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

## 1. Coleta Dinâmica dos CSVs de Métricas

In [ ]:
MODELS_DIR = os.path.join(os.path.dirname(os.getcwd()), 'models')

# Mapeamento de modelos para tipo de problema
MODELOS_REGRESSAO = {'regressao_linear', 'regressao_linear_rede_neural'}
MODELOS_CLASSIFICACAO = {'regressao_logistica', 'random_forest', 'xgboost'}

# Nomes amigáveis para exibição
NOMES_MODELOS = {
    'regressao_linear': 'Regressão Linear',
    'regressao_linear_rede_neural': 'Rede Neural (Regressão)',
    'regressao_logistica': 'Regressão Logística',
    'random_forest': 'Random Forest',
    'xgboost': 'XGBoost',
}

NOMES_DATASETS = {
    'dataset_base': 'Base',
    'dataset_dummy': 'Dummy',
    'dataset_indicadores': 'Indicadores',
    'dataset_janelas': 'Janelas',
}

HORIZONTES = ['3 dias', '7 dias', '15 dias', '30 dias']

print(f'Diretório dos modelos: {MODELS_DIR}')
print(f'Existe: {os.path.exists(MODELS_DIR)}')

In [ ]:
def coletar_metricas(models_dir):
    """
    Varre dinamicamente o diretório de modelos e coleta todos os CSVs de métricas.
    Retorna um DataFrame consolidado com colunas extras:
    - modelo, dataset, ticker, tipo (regressao/classificacao)
    """
    registros = []
    
    csv_files = glob.glob(os.path.join(models_dir, '**', '*.csv'), recursive=True)
    print(f'Total de arquivos CSV encontrados: {len(csv_files)}')
    
    for csv_path in csv_files:
        # Extrair informações do caminho
        # Estrutura: models/<modelo>/<dataset>/<subpasta>/metricas_<ticker>_<sufixo>.csv
        rel_path = os.path.relpath(csv_path, models_dir)
        parts = rel_path.split(os.sep)
        
        if len(parts) < 3:
            continue
        
        modelo = parts[0]       # ex: regressao_linear
        dataset = parts[1]      # ex: dataset_janelas
        
        # Extrair ticker do nome do arquivo
        filename = os.path.basename(csv_path)
        ticker = None
        for t in ['agro3', 'slce3', 'soja3']:
            if t in filename.lower():
                ticker = t.upper()
                break
        
        if ticker is None:
            continue
        
        # Determinar tipo
        if modelo in MODELOS_REGRESSAO:
            tipo = 'regressao'
        elif modelo in MODELOS_CLASSIFICACAO:
            tipo = 'classificacao'
        else:
            continue
        
        # Ler CSV
        try:
            df = pd.read_csv(csv_path)
            # Normalizar nome da coluna R² / R2
            df.columns = [c.replace('R²', 'R2') for c in df.columns]
            
            df['modelo'] = modelo
            df['modelo_nome'] = NOMES_MODELOS.get(modelo, modelo)
            df['dataset'] = dataset
            df['dataset_nome'] = NOMES_DATASETS.get(dataset, dataset)
            df['ticker'] = ticker
            df['tipo'] = tipo
            
            registros.append(df)
        except Exception as e:
            print(f'Erro ao ler {csv_path}: {e}')
    
    if not registros:
        print('Nenhum registro encontrado!')
        return pd.DataFrame()
    
    df_all = pd.concat(registros, ignore_index=True)
    print(f'Total de registros coletados: {len(df_all)}')
    print(f'Modelos encontrados: {sorted(df_all["modelo"].unique())}')
    print(f'Datasets encontrados: {sorted(df_all["dataset"].unique())}')
    print(f'Tickers encontrados: {sorted(df_all["ticker"].unique())}')
    print(f'Tipos: {sorted(df_all["tipo"].unique())}')
    return df_all

df_all = coletar_metricas(MODELS_DIR)
df_all.head()

## 2. Separação por Tipo de Problema

In [ ]:
df_regressao = df_all[df_all['tipo'] == 'regressao'].copy()
df_classificacao = df_all[df_all['tipo'] == 'classificacao'].copy()

print(f'Registros de Regressão: {len(df_regressao)}')
print(f'Registros de Classificação: {len(df_classificacao)}')

---
## 3. Rankings de REGRESSÃO (por Janela de Tempo)

Métricas de regressão:
- **R²** (quanto maior, melhor)
- **MAE** (quanto menor, melhor)
- **RMSE** (quanto menor, melhor)

O ranking principal é ordenado por **R² decrescente**.

In [ ]:
def ranking_regressao(df, horizonte):
    """
    Gera ranking de modelos de regressão para um horizonte específico.
    Ordenado por R² (decrescente), depois MAE (crescente).
    """
    df_h = df[df['Horizonte'] == horizonte].copy()
    if df_h.empty:
        print(f'  Sem dados para horizonte: {horizonte}')
        return pd.DataFrame()
    
    df_h = df_h.sort_values(['R2', 'MAE'], ascending=[False, True]).reset_index(drop=True)
    df_h.index = df_h.index + 1
    df_h.index.name = 'Ranking'
    
    cols = ['modelo_nome', 'dataset_nome', 'ticker', 'R2', 'MAE', 'RMSE']
    return df_h[cols].rename(columns={
        'modelo_nome': 'Modelo',
        'dataset_nome': 'Dataset',
        'ticker': 'Ticker',
    })


for horizonte in HORIZONTES:
    print(f'\n{"="*80}')
    print(f'  RANKING REGRESSÃO — {horizonte.upper()}')
    print(f'{"="*80}')
    ranking = ranking_regressao(df_regressao, horizonte)
    if not ranking.empty:
        display(ranking)

---
## 4. Rankings de CLASSIFICAÇÃO (por Janela de Tempo)

Métricas de classificação:
- **Accuracy, Precision, Recall, F1-Score, AUC-ROC** (quanto maior, melhor)

O ranking principal é ordenado por **F1-Score decrescente**.

In [ ]:
def ranking_classificacao(df, horizonte):
    """
    Gera ranking de modelos de classificação para um horizonte específico.
    Ordenado por F1-Score (decrescente), depois AUC-ROC (decrescente).
    """
    df_h = df[df['Horizonte'] == horizonte].copy()
    if df_h.empty:
        print(f'  Sem dados para horizonte: {horizonte}')
        return pd.DataFrame()
    
    df_h = df_h.sort_values(['F1-Score', 'AUC-ROC'], ascending=[False, False]).reset_index(drop=True)
    df_h.index = df_h.index + 1
    df_h.index.name = 'Ranking'
    
    cols = ['modelo_nome', 'dataset_nome', 'ticker', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
    return df_h[cols].rename(columns={
        'modelo_nome': 'Modelo',
        'dataset_nome': 'Dataset',
        'ticker': 'Ticker',
    })


for horizonte in HORIZONTES:
    print(f'\n{"="*80}')
    print(f'  RANKING CLASSIFICAÇÃO — {horizonte.upper()}')
    print(f'{"="*80}')
    ranking = ranking_classificacao(df_classificacao, horizonte)
    if not ranking.empty:
        display(ranking)

---
## 5. Ranking Médio por Modelo (agregado por dataset e ticker)

Média das métricas de cada modelo em todos os datasets e tickers, segmentado por janela de tempo.

In [ ]:
print('=' * 80)
print('  RANKING MÉDIO — REGRESSÃO (média por modelo e horizonte)')
print('=' * 80)

for horizonte in HORIZONTES:
    df_h = df_regressao[df_regressao['Horizonte'] == horizonte]
    if df_h.empty:
        continue
    
    media = (
        df_h.groupby('modelo_nome')[['R2', 'MAE', 'RMSE']]
        .mean()
        .sort_values('R2', ascending=False)
    )
    media.index.name = 'Modelo'
    
    print(f'\n--- {horizonte} ---')
    display(media)

In [ ]:
print('=' * 80)
print('  RANKING MÉDIO — CLASSIFICAÇÃO (média por modelo e horizonte)')
print('=' * 80)

for horizonte in HORIZONTES:
    df_h = df_classificacao[df_classificacao['Horizonte'] == horizonte]
    if df_h.empty:
        continue
    
    media = (
        df_h.groupby('modelo_nome')[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']]
        .mean()
        .sort_values('F1-Score', ascending=False)
    )
    media.index.name = 'Modelo'
    
    print(f'\n--- {horizonte} ---')
    display(media)

---
## 6. Visualizações Comparativas

In [ ]:
# ============================================================
# GRÁFICO: R² médio por modelo de regressão e horizonte
# ============================================================

fig, axes = plt.subplots(1, 4, figsize=(22, 5), sharey=True)
fig.suptitle('R² Médio por Modelo de Regressão — Segmentado por Horizonte', fontsize=15, fontweight='bold')

cores_regressao = ['#2196F3', '#FF9800']

for i, horizonte in enumerate(HORIZONTES):
    ax = axes[i]
    df_h = df_regressao[df_regressao['Horizonte'] == horizonte]
    if df_h.empty:
        ax.set_title(horizonte)
        continue
    
    media = df_h.groupby('modelo_nome')['R2'].mean().sort_values(ascending=True)
    bars = ax.barh(media.index, media.values, color=cores_regressao[:len(media)])
    ax.set_title(horizonte, fontsize=13, fontweight='bold')
    ax.set_xlim(0, 1)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    
    for bar, val in zip(bars, media.values):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.4f}',
                va='center', fontsize=10)

axes[0].set_xlabel('R²')
plt.tight_layout()
plt.savefig('ranking_regressao_r2.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# GRÁFICO: F1-Score médio por modelo de classificação e horizonte
# ============================================================

fig, axes = plt.subplots(1, 4, figsize=(22, 5), sharey=True)
fig.suptitle('F1-Score Médio por Modelo de Classificação — Segmentado por Horizonte', fontsize=15, fontweight='bold')

cores_classificacao = ['#4CAF50', '#E91E63', '#9C27B0']

for i, horizonte in enumerate(HORIZONTES):
    ax = axes[i]
    df_h = df_classificacao[df_classificacao['Horizonte'] == horizonte]
    if df_h.empty:
        ax.set_title(horizonte)
        continue
    
    media = df_h.groupby('modelo_nome')['F1-Score'].mean().sort_values(ascending=True)
    bars = ax.barh(media.index, media.values, color=cores_classificacao[:len(media)])
    ax.set_title(horizonte, fontsize=13, fontweight='bold')
    ax.set_xlim(0, 1)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    
    for bar, val in zip(bars, media.values):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.4f}',
                va='center', fontsize=10)

axes[0].set_xlabel('F1-Score')
plt.tight_layout()
plt.savefig('ranking_classificacao_f1.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# GRÁFICO: Evolução das métricas por horizonte (linha)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Regressão: R² por horizonte ---
ax = axes[0]
ax.set_title('Evolução do R² Médio por Horizonte — Regressão', fontsize=13, fontweight='bold')

for modelo in sorted(df_regressao['modelo_nome'].unique()):
    df_m = df_regressao[df_regressao['modelo_nome'] == modelo]
    medias = []
    for h in HORIZONTES:
        val = df_m[df_m['Horizonte'] == h]['R2'].mean()
        medias.append(val)
    ax.plot([3, 7, 15, 30], medias, marker='o', linewidth=2, label=modelo)

ax.set_xlabel('Horizonte (dias)')
ax.set_ylabel('R² Médio')
ax.set_xticks([3, 7, 15, 30])
ax.legend()
ax.grid(True, alpha=0.3)

# --- Classificação: F1-Score por horizonte ---
ax = axes[1]
ax.set_title('Evolução do F1-Score Médio por Horizonte — Classificação', fontsize=13, fontweight='bold')

for modelo in sorted(df_classificacao['modelo_nome'].unique()):
    df_m = df_classificacao[df_classificacao['modelo_nome'] == modelo]
    medias = []
    for h in HORIZONTES:
        val = df_m[df_m['Horizonte'] == h]['F1-Score'].mean()
        medias.append(val)
    ax.plot([3, 7, 15, 30], medias, marker='s', linewidth=2, label=modelo)

ax.set_xlabel('Horizonte (dias)')
ax.set_ylabel('F1-Score Médio')
ax.set_xticks([3, 7, 15, 30])
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('evolucao_metricas_horizonte.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 7. Ranking por Ticker (detalhamento)

In [ ]:
tickers = sorted(df_all['ticker'].unique())

print('=' * 80)
print('  RANKING POR TICKER — REGRESSÃO')
print('=' * 80)

for ticker in tickers:
    for horizonte in HORIZONTES:
        df_h = df_regressao[(df_regressao['ticker'] == ticker) & (df_regressao['Horizonte'] == horizonte)]
        if df_h.empty:
            continue
        
        df_h = df_h.sort_values('R2', ascending=False).reset_index(drop=True)
        df_h.index = df_h.index + 1
        df_h.index.name = 'Ranking'
        
        print(f'\n--- {ticker} | {horizonte} ---')
        display(df_h[['modelo_nome', 'dataset_nome', 'R2', 'MAE', 'RMSE']].rename(columns={
            'modelo_nome': 'Modelo', 'dataset_nome': 'Dataset'
        }))

In [ ]:
print('=' * 80)
print('  RANKING POR TICKER — CLASSIFICAÇÃO')
print('=' * 80)

for ticker in tickers:
    for horizonte in HORIZONTES:
        df_h = df_classificacao[(df_classificacao['ticker'] == ticker) & (df_classificacao['Horizonte'] == horizonte)]
        if df_h.empty:
            continue
        
        df_h = df_h.sort_values('F1-Score', ascending=False).reset_index(drop=True)
        df_h.index = df_h.index + 1
        df_h.index.name = 'Ranking'
        
        print(f'\n--- {ticker} | {horizonte} ---')
        display(df_h[['modelo_nome', 'dataset_nome', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']].rename(columns={
            'modelo_nome': 'Modelo', 'dataset_nome': 'Dataset'
        }))

---
## 8. Heatmaps: Desempenho por Modelo × Dataset

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Heatmap R² — Regressão (Modelo × Dataset)', fontsize=15, fontweight='bold')

for i, horizonte in enumerate(HORIZONTES):
    ax = axes[i // 2][i % 2]
    df_h = df_regressao[df_regressao['Horizonte'] == horizonte]
    
    if df_h.empty:
        ax.set_title(f'{horizonte} (sem dados)')
        continue
    
    pivot = df_h.pivot_table(
        values='R2', index='modelo_nome', columns='dataset_nome', aggfunc='mean'
    )
    
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='RdYlGn', ax=ax,
                vmin=0, vmax=1, linewidths=0.5)
    ax.set_title(f'{horizonte}', fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.tight_layout()
plt.savefig('heatmap_regressao_r2.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Heatmap F1-Score — Classificação (Modelo × Dataset)', fontsize=15, fontweight='bold')

for i, horizonte in enumerate(HORIZONTES):
    ax = axes[i // 2][i % 2]
    df_h = df_classificacao[df_classificacao['Horizonte'] == horizonte]
    
    if df_h.empty:
        ax.set_title(f'{horizonte} (sem dados)')
        continue
    
    pivot = df_h.pivot_table(
        values='F1-Score', index='modelo_nome', columns='dataset_nome', aggfunc='mean'
    )
    
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='RdYlGn', ax=ax,
                vmin=0, vmax=1, linewidths=0.5)
    ax.set_title(f'{horizonte}', fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.tight_layout()
plt.savefig('heatmap_classificacao_f1.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 9. Resumo Final: Melhor Modelo por Horizonte

In [ ]:
print('=' * 80)
print('  MELHOR MODELO POR HORIZONTE')
print('=' * 80)

resumo_regressao = []
resumo_classificacao = []

for horizonte in HORIZONTES:
    # Regressão: melhor R²
    df_h = df_regressao[df_regressao['Horizonte'] == horizonte]
    if not df_h.empty:
        best = df_h.loc[df_h['R2'].idxmax()]
        resumo_regressao.append({
            'Horizonte': horizonte,
            'Modelo': best['modelo_nome'],
            'Dataset': best['dataset_nome'],
            'Ticker': best['ticker'],
            'R2': best['R2'],
            'MAE': best['MAE'],
            'RMSE': best['RMSE'],
        })
    
    # Classificação: melhor F1-Score
    df_h = df_classificacao[df_classificacao['Horizonte'] == horizonte]
    if not df_h.empty:
        best = df_h.loc[df_h['F1-Score'].idxmax()]
        resumo_classificacao.append({
            'Horizonte': horizonte,
            'Modelo': best['modelo_nome'],
            'Dataset': best['dataset_nome'],
            'Ticker': best['ticker'],
            'Accuracy': best['Accuracy'],
            'F1-Score': best['F1-Score'],
            'AUC-ROC': best['AUC-ROC'],
        })

print('\n--- Regressão (melhor R²) ---')
display(pd.DataFrame(resumo_regressao).set_index('Horizonte'))

print('\n--- Classificação (melhor F1-Score) ---')
display(pd.DataFrame(resumo_classificacao).set_index('Horizonte'))

---
## 10. Exportar Rankings Consolidados

In [ ]:
# Exportar CSVs consolidados para análise posterior
output_dir = os.path.dirname(os.path.abspath('__file__'))

df_regressao.to_csv(os.path.join(output_dir, 'consolidado_regressao.csv'), index=False)
df_classificacao.to_csv(os.path.join(output_dir, 'consolidado_classificacao.csv'), index=False)

# Rankings médios por modelo e horizonte
ranking_medio_reg = []
ranking_medio_clf = []

for horizonte in HORIZONTES:
    # Regressão
    df_h = df_regressao[df_regressao['Horizonte'] == horizonte]
    if not df_h.empty:
        media = df_h.groupby('modelo_nome')[['R2', 'MAE', 'RMSE']].mean().reset_index()
        media['Horizonte'] = horizonte
        ranking_medio_reg.append(media)
    
    # Classificação
    df_h = df_classificacao[df_classificacao['Horizonte'] == horizonte]
    if not df_h.empty:
        media = df_h.groupby('modelo_nome')[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']].mean().reset_index()
        media['Horizonte'] = horizonte
        ranking_medio_clf.append(media)

if ranking_medio_reg:
    pd.concat(ranking_medio_reg).to_csv(os.path.join(output_dir, 'ranking_medio_regressao.csv'), index=False)

if ranking_medio_clf:
    pd.concat(ranking_medio_clf).to_csv(os.path.join(output_dir, 'ranking_medio_classificacao.csv'), index=False)

print('Arquivos exportados com sucesso:')
print('  - consolidado_regressao.csv')
print('  - consolidado_classificacao.csv')
print('  - ranking_medio_regressao.csv')
print('  - ranking_medio_classificacao.csv')